<a href="https://colab.research.google.com/github/Davidson-Open-University/csc203-labs/blob/main/notebooks/CSC203-Lab2-Testing-Claims-and-Checking-Proofs.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CSC 203 Lab 2: Testing Claims and Checking Proofs

**Study Session 2: Proofs and Induction** · Davidson Open University Global

**In this lab you will:**
- test a mathematical claim on many values and find counterexamples;
- explain why examples are evidence but not a proof;
- use SymPy to check the algebra in the inductive step of a proof by induction;
- write functions that test primes and factorise numbers.

**Time:** about 60 minutes in four parts:
- **Part A – Explore** (about 10 minutes): run the examples and read the explanations.
- **Part B – Practise** (about 20 minutes): fill in the blanks. A self-check tells you if you are right.
- **Part C – Build a tool** (about 20 minutes): complete a short Python function and test it.
- **Part D – Apply and reflect** (about 10 minutes): use your tools on your own example and write short answers.

**How to use this notebook**
1. Run each code cell in order: click it and press **Shift + Enter** (or the ▶ button).
2. Where you see `...`, replace it with your own code, run the cell, then run the **check** cell under it.
   - ✅ means correct. ❌ gives you a hint: try again. ⏳ means you have not filled in the blank yet.
   - The checks never show the answer, so you learn by working it out. You can try as many times as you like.
3. Stuck? Read the hint, look back at the study session, or ask in the discussion forum. Do not skip ahead: each part builds on the last.
4. Save your work: in Colab choose **File → Save a copy in GitHub** and save it to your own `csc203-labs` repository. Offline in Jupyter, press **Ctrl + S** and later upload the file with **Add file → Upload files** on GitHub.

*This lab is practice to help you learn and prepare for the assignment. It is not graded unless your tutor tells you otherwise.*

In [ ]:
#@title ▶ Run this cell first: it sets up the self-checks (you do not need to read it)
# You do not need to read or change this code.
import hashlib as _h
class _Blank(Exception): pass
def _norm(x, unordered=False):
    try:
        import numpy as _np
        if isinstance(x, _np.ndarray): x = x.tolist()
        elif isinstance(x, _np.bool_): x = bool(x)
        elif isinstance(x, _np.integer): x = int(x)
        elif isinstance(x, _np.floating): x = float(x)
    except ImportError:
        pass
    if x is Ellipsis or x is None: raise _Blank()
    if isinstance(x, bool): return 'b:' + str(x)
    if isinstance(x, (int, float)): return 'n:' + repr(round(float(x), 6) + 0.0)
    if isinstance(x, str): return 't:' + ' '.join(x.lower().split())
    if isinstance(x, (set, frozenset)): return 's{' + ','.join(sorted(_norm(i) for i in x)) + '}'
    if isinstance(x, (list, tuple)):
        items = [_norm(i) for i in x]
        if unordered: items.sort()
        return 'l[' + ','.join(items) + ']'
    if isinstance(x, dict): return 'd{' + ','.join(sorted(_norm(k) + '=' + _norm(v) for k, v in x.items())) + '}'
    try:
        import sympy as _sp
        if isinstance(x, _sp.Basic):
            x = _sp.sympify(x)
            if x.is_number: return 'n:' + repr(round(float(x), 6) + 0.0)
            syms = sorted(x.free_symbols, key=str)
            vals = []
            for t in (0.3, 0.7, 1.3):
                v = complex(x.subs({s: t + 0.11 * i for i, s in enumerate(syms)}).evalf())
                vals.append(round(v.real, 6) + 0.0)
            return 'e:' + repr(vals)
    except ImportError:
        pass
    return 'r:' + repr(x)
def _hash(q, s): return _h.sha256(('csc203|' + q + '|' + s).encode()).hexdigest()[:16]
def check(q, value):
    """Check your answer to question q. Prints ✅, ❌ with a hint, or ⏳ if the blank is not filled yet."""
    spec = _CHECKS[q]
    try:
        if callable(value) and 'args' in spec:
            outs = []
            for a in spec['args']:
                try:
                    outs.append(value(*a))
                except _Blank:
                    raise
                except Exception as e:
                    print(f'❌ Question {q}: your function stopped with an error: {type(e).__name__}: {e}')
                    return
            if all(o is None or o is Ellipsis for o in outs): raise _Blank()
            got = _norm(outs, False)
        elif spec.get('logic'):
            if value is None or value is Ellipsis: raise _Blank()
            import sympy as _sp
            from itertools import product as _p
            syms = sorted(_sp.sympify(value).free_symbols, key=str)
            got = 'L' + ''.join('1' if bool(_sp.sympify(value).subs(dict(zip(syms, v)))) else '0' for v in _p([True, False], repeat=len(syms))) + ':' + ','.join(map(str, syms))
        else:
            got = _norm(value, spec.get('unordered', False))
    except _Blank:
        print(f'⏳ Question {q}: fill in the blank (replace ...) and run the cell above, then run this cell again.')
        return
    if _RECORD is not None:
        _RECORD[q] = _hash(q, got); print(f'✅ Question {q}: recorded.'); return
    if _hash(q, got) == spec['h']:
        print(f'✅ Question {q}: correct! ' + spec.get('ok', 'Well done.'))
    else:
        print(f'❌ Question {q}: not yet. Hint: ' + spec['hint'])
_RECORD = None

_CHECKS = {
    '2.1': {'h': 'dca27c46c89c6fb7', 'hint': 'sum(range(start, stop, step)) adds the numbers. The stop value is not included.', 'ok': 'Compare with 100²: the sum of the first n odd numbers is n².'},
    '2.2': {'h': 'a9b0e84c1c915244', 'hint': 'Use for n in range(1, 1000): ... if not is_prime_demo(n*n - n + 41): store n and break.', 'ok': 'Try factorising the value at that n: why does 41 divide it?'},
    '2.3': {'h': '35a57707af95bad2', 'hint': 'Loop n = 0, 1, 2, ... and test 2 ** (2 ** n) + 1. The counterexample is less than 7.', 'ok': 'Leonhard Euler found this counterexample in 1732: it equals 641 × 6700417. Even great mathematicians need proofs!'},
    '2.4': {'h': '3fb27cb140432439', 'hint': 'Type the formula with * for multiplication: n * (n + 1) * (2 * n + 1) / 6.', 'ok': 'The inductive step works, and G(1) = 1 checks the base case.'},
    '2.5': {'h': 'b5f029812562b3d0', 'hint': 'Check the special cases: 0 and 1 are not prime, 2 is prime, and a square such as 49 = 7 × 7 is not prime (test d * d <= m, not d * d < m).', 'ok': 'Your prime test passes all the test cases.', 'args': [(0,), (1,), (2,), (3,), (4,), (9,), (49,), (97,), (91,), (7919,)]},
    '2.6': {'h': 'f3ab66151b9afbaa', 'hint': 'Use a while loop: while n > 1. Only increase d when it does NOT divide n, so that repeated factors like 2, 2 are found.', 'ok': 'This is the Fundamental Theorem of Arithmetic in action.', 'args': [(60,), (2,), (97,), (1024,), (2 ** 32 + 1,), (360,)]},
}
print('Self-checks ready. Now carry on with Part A.')

## Part A – Explore (about 10 minutes)
A computer cannot prove a statement about **every** natural number, because there are infinitely many. But it can:
- **test** a claim on thousands of values, which gives *evidence*;
- find a **counterexample**, which *disproves* a claim straight away;
- check the **algebra** in a proof, so that you do not make slips.

Run the cell: it tests the formula 1 + 2 + … + n = n(n + 1)/2 for n = 1 to 1000.

In [ ]:
def sum_to(n):
    return sum(range(1, n + 1))          # 1 + 2 + ... + n

ok = all(sum_to(n) == n * (n + 1) // 2 for n in range(1, 1001))
print('Formula true for n = 1 to 1000?', ok)

**Passing tests is not a proof.** The famous claim "n² + n + 41 is prime for every n ≥ 0" (Euler, 1772) works for n = 0, 1, 2, …, 39. **Predict:** do you think it is true for all n? Run the cell.

In [ ]:
def is_prime_demo(m):
    return m > 1 and all(m % d != 0 for d in range(2, int(m ** 0.5) + 1))

for n in range(0, 100):
    value = n * n + n + 41
    if not is_prime_demo(value):
        print(f'Counterexample: n = {n} gives {value} = 41 × {value // 41}')
        break
else:
    print('No counterexample below 100')

Forty successes, then a failure. This is why mathematicians need **proof**: a proof by induction shows the result for *every* n at once.

**How induction works (recap).** To prove S(n) for all n ≥ 1:
1. **Base case:** show S(1) is true.
2. **Inductive step:** assume S(k) is true (the *inductive hypothesis*) and show S(k + 1) follows.

**SymPy** is a free Python library for algebra. It treats `k` as a symbol, not a number. Below, it checks the inductive step for 1 + 2 + … + n = n(n + 1)/2: adding the next term (k + 1) to the formula for k must give the formula for k + 1.

In [ ]:
import sympy as sp
k = sp.symbols('k', integer=True, positive=True)

def F(n):
    return n * (n + 1) / 2          # the claimed formula

difference = F(k) + (k + 1) - F(k + 1)
print('F(k) + (k + 1) - F(k + 1) simplifies to:', sp.simplify(difference))
print('Base case F(1) =', F(1))

The difference simplifies to **0**, so the inductive step is algebraically correct.

## Part B – Practise (about 20 minutes)
**B1. Sum of odd numbers.** Use `sum` and `range` to find 1 + 3 + 5 + … + 199 (the first 100 odd numbers). *Hint:* `range(1, 200, 2)` counts 1, 3, 5, …, 199.

In [ ]:
answer_2_1 = ...

In [ ]:
check('2.1', answer_2_1)

**B2. Another counterexample.** The claim "n² − n + 41 is prime for every n ≥ 1" is also false. Use a loop (copy the pattern from Part A, using `is_prime_demo`) to find the **smallest** n ≥ 1 for which n² − n + 41 is not prime.

In [ ]:
answer_2_2 = ...
# YOUR CODE: loop over n = 1, 2, 3, ... and store the first n where n*n - n + 41 is not prime

In [ ]:
check('2.2', answer_2_2)

**B3. A claim by a famous mathematician.** In 1640 Pierre de Fermat claimed that every number of the form 2^(2ⁿ) + 1 is prime. Find the **smallest n ≥ 0** for which it is *not* prime. (Python writes powers with `**`, so 2^(2ⁿ) is `2 ** (2 ** n)`. `is_prime_demo` is fast enough for these numbers.)

In [ ]:
answer_2_3 = ...

In [ ]:
check('2.3', answer_2_3)

**B4. Check an inductive step with SymPy.** The claim is 1² + 2² + … + n² = n(n + 1)(2n + 1)/6.
Write the formula as a function `G(n)`, then compute the difference G(k) + (k + 1)² − G(k + 1), exactly as in Part A. If the inductive step is correct, it simplifies to 0.

In [ ]:
def G(n):
    return ...        # YOUR CODE: the formula n(n + 1)(2n + 1)/6

answer_2_4 = sp.simplify(G(k) + (k + 1) ** 2 - G(k + 1)) if G(1) is not ... else ...
print(answer_2_4)

In [ ]:
check('2.4', answer_2_4)

## Part C – Build a tool (about 20 minutes)
### C1. A prime test
A number m is **prime** if m > 1 and no number d with 2 ≤ d ≤ √m divides it. Complete the function: return `False` for m ≤ 1; then loop over d and return `False` as soon as `m % d == 0`; if no divisor is found, return `True`.

In [ ]:
def is_prime(m):
    # YOUR CODE
    ...

print(is_prime(2), is_prime(15), is_prime(97))

In [ ]:
check('2.5', is_prime)

### C2. Every number is a product of primes
**Strong induction** proves that every integer n ≥ 2 can be written as a product of primes. The proof gives a method: find the smallest divisor d > 1 of n (it must be prime), divide it out, and repeat.

Complete `factorise(n)` so that it returns the list of prime factors in increasing order, for example `factorise(60)` → `[2, 2, 3, 5]`.

In [ ]:
def factorise(n):
    d = 2
    factors = ...    # YOUR CODE: start with an empty list []. Then, while n > 1: if d divides n,
                     # add d to factors and divide n by d (n = n // d); otherwise increase d by 1.
    return factors

print(factorise(60))

In [ ]:
check('2.6', factorise)

## Part D – Apply and reflect (about 10 minutes)
**D1. Prepare for Assignment 2.** The assignment asks you to prove 2 + 4 + 6 + … + 2n = n(n + 1) by induction.
In the cell below, (a) test the claim for n = 1 to 500, and (b) use SymPy to check that your inductive step simplifies to 0. A computer check does **not** replace the written proof: you still need to write the base case, the hypothesis and the step in words and symbols.

In [ ]:
# D1 (a): test the claim for n = 1 to 500


# D1 (b): check the inductive step with SymPy

**D2. Reflect.** Answer in one or two sentences each:
1. The claim in B2 is true for n = 1 to 40. Why does that still not prove it?
2. In the inductive step you *assume* S(k) is true. Why is that not "assuming what you want to prove"?
3. What does your `factorise` function have to do with strong induction?

*Type your answer here (double-click this cell to edit it).*

## Finished? Save your work
In Colab: **File → Save a copy in GitHub** → your `csc203-labs` repository → commit message *"Lab 2 completed"*.
Offline: save with **Ctrl + S**, then upload the file to your repository with **Add file → Upload files** when you are online.

Then go back to Moodle and mark the lab as done.